## OpenAI com skill prompts

Esta secao usa os prompts da skill `ecossistema-agentes-prompts` para orquestrador, especialistas e avaliador.

In [12]:
import os
import re
import json
import sys
from typing import Dict, List, Optional, Tuple
from openai import OpenAI
from dotenv import load_dotenv
import logging

load_dotenv()

# Logger do Maestro — troque para logging.DEBUG para ver detalhes de execução
logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s"
)
logger = logging.getLogger("maestro")

sys.path.insert(0, os.path.abspath("."))

In [13]:
# Caminho base das skills (pode ser absoluto ou relativo ao notebook)
SKILLS_DIR = "mnt/skills"
SKILL_FILE = "SKILL.md"

def _parse_frontmatter(content: str) -> Tuple[Dict, str]:
    """Extrai frontmatter YAML (entre ---) e retorna (metadados, resto do conteúdo)."""
    meta = {}
    body = content
    match = re.match(r"^---\s*\n(.*?)\n---\s*\n(.*)", content, re.DOTALL)
    if match:
        fm, body = match.group(1), match.group(2)
        for line in fm.split("\n"):
            if ":" in line:
                key, val = line.split(":", 1)
                meta[key.strip()] = val.strip().strip("'\"").replace("\n", " ").strip()
    return meta, body.strip()

def load_skills(base_dir: str) -> List[Dict]:
    """
    Carrega todas as skills a partir de base_dir.
    Cada subpasta contendo SKILL.md vira uma skill.
    Retorna lista de dicts: id, name, description, content (texto completo).
    """
    skills = []
    base = os.path.abspath(base_dir)
    if not os.path.isdir(base):
        return skills
    for entry in sorted(os.listdir(base)):
        path = os.path.join(base, entry)
        skill_file = os.path.join(path, SKILL_FILE)
        if os.path.isdir(path) and os.path.isfile(skill_file):
            with open(skill_file, "r", encoding="utf-8") as f:
                raw = f.read()
            meta, body = _parse_frontmatter(raw)
            skill_id = entry
            name = meta.get("name", skill_id)
            description = meta.get("description", "")
            skills.append({
                "id": skill_id,
                "name": name,
                "description": description,
                "content": raw,
                "body": body,
            })
    return skills

def get_skill_by_id(skills: List[Dict], skill_id: str) -> Optional[Dict]:
    """Retorna a skill com o id dado ou None."""
    for s in skills:
        if s["id"] == skill_id:
            return s
    return None

# Carrega todas as skills
skills_loaded = load_skills(SKILLS_DIR)
print(f"Skills carregadas: {len(skills_loaded)}")
for s in skills_loaded:
    print(f"  - {s['id']}: {s.get('name', s['id'])}")
    # print(s['content'])

Skills carregadas: 11
  - agente-cientifico: agente-cientifico
  - agente-dados: agente-dados
  - agente-financeiro: agente-financeiro
  - agente-juridico: agente-juridico
  - agente-mercado: agente-mercado
  - agente-mysql: agente-mysql
  - agente-negocios: agente-negocios
  - agente-saude: agente-saude
  - agente-tecnico: agente-tecnico
  - avaliador-coerencia: avaliador-coerencia
  - maestro: maestro


In [14]:
## Defini client e o modelo e carregar as skills
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
model = os.environ.get("MODELO_DEFAULT")
skills = load_skills("mnt/skills")

In [15]:
def chat_with_skill(
    client: OpenAI,
    skill_id: str,
    user_message: str,
    model: str,
    skills_list: Optional[List[Dict]] = None,
) -> str:
    """
    Envia uma mensagem ao modelo OpenAI usando o conteúdo da skill como system prompt.
    Se skills_list for None, usa skills_loaded (carregadas na célula anterior).
    """
    skills = skills_list if skills_list is not None else skills_loaded
    skill = get_skill_by_id(skills, skill_id)
    if not skill:
        return f"Erro: skill '{skill_id}' não encontrada."
    system = skill["content"]
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_message},
        ],
    )
    return resp.choices[0].message.content or ""

# Exemplo:
# client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
# model = os.environ.get("MODELO_DEFAULT")
# resposta = chat_with_skill(client, "agente-tecnico", "Como implementar um RAG com embeddings?", model)
# print(resposta)

In [16]:
def extrair_json(texto: str) -> Optional[Dict]:
    """Extrai JSON do texto retornado pelo modelo (pode vir em bloco ```json ... ```)."""
    if not texto or not texto.strip():
        return None
    s = texto.strip()
    # Remove bloco markdown de código
    if "```" in s:
        match = re.search(r"```(?:json)?\s*([\s\S]*?)```", s)
        if match:
            s = match.group(1).strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        return None

def invocar_agente_maestro(
    client: OpenAI,
    skill_id: str,
    payload_maestro: Dict,
    model: str,
    skills_list: Optional[List[Dict]] = None,
) -> str:
    """
    Invoca um agente em modo Maestro; o modelo deve retornar JSON no Formato de Retorno da skill.
    Retorna o texto bruto da resposta (parse com extrair_json depois).
    """
    skills = skills_list if skills_list is not None else skills_loaded
    skill = get_skill_by_id(skills, skill_id)
    if not skill:
        return ""
    system = skill["content"]
    user = (
        json.dumps(payload_maestro, ensure_ascii=False)
        + "\n\nVocê está sendo invocado pelo Maestro. Responda APENAS com um único JSON válido no Formato de Retorno da skill "
        "(agente_id, agente_nome, pode_responder, justificativa_viabilidade, resposta, scores, limitacoes_da_resposta, aspectos_para_outros_agentes). "
        "Sem texto antes ou depois."
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        response_format={"type": "json_object"},
    )
    return (resp.choices[0].message.content or "").strip()

In [17]:
def executar_fluxo_maestro(
    client: OpenAI,
    pergunta: str,
    model: str = None,
    skills_list: Optional[List[Dict]] = None,
    agentes: Optional[List[str]] = None,
    agentes_dataframe: Optional[List[str]] = None,
    verbose: bool = True,
    mysql_host: Optional[str] = None,
    mysql_porta: Optional[int] = None,
    mysql_usuario: Optional[str] = None,
    mysql_senha: Optional[str] = None,
    mysql_banco: Optional[str] = None,
    mysql_tabela: Optional[str] = None,
    mysql_limite: int = 50_000,
    mysql_filtro_where: str = "",
    mysql_injetar_namespace: Optional[Dict] = None,
) -> Dict:
    """
    Executa o fluxo completo: análise (1+2), invocação dos N agentes (3),
    payload avaliador (4), avaliador (5), entrega (6).

    Parâmetros novos:
        agentes_dataframe: lista de skill_ids que devem receber df_contexto
                           quando mysql_tabela for fornecida.
                           Padrão: ["agente-dados", "agente-financeiro", "agente-negocios"]

    Retorna dict com analise, respostas_agentes, avaliacao, entrega_final.
    """
    model = model or os.environ.get("MODELO_DEFAULT")
    skills = skills_list if skills_list is not None else skills_loaded
    agentes = agentes or []
    _agentes_dataframe = agentes_dataframe if agentes_dataframe is not None \
        else ["agente-dados", "agente-financeiro", "agente-negocios"]
    resultado_mysql = None
    df_contexto = None
    df_variavel = None

    def _preparar_contexto_mysql() -> Tuple[Dict, Dict]:
        from mnt.skills.agente_mysql.helpers import MySQLAgent

        host     = mysql_host     or os.environ.get("MYSQL_HOST", "localhost")
        porta    = mysql_porta
        if porta is None:
            porta_env = os.environ.get("MYSQL_PORT")
            porta = int(porta_env) if porta_env else 3306
        usuario  = mysql_usuario  or os.environ.get("MYSQL_USER", "root")
        senha    = mysql_senha    or os.environ.get("MYSQL_PASSWORD", "")
        banco    = mysql_banco    or os.environ.get("MYSQL_DATABASE", "")

        agent = MySQLAgent(host=host, porta=porta, usuario=usuario, senha=senha, banco=banco)
        resultado = agent.carregar_tabela(
            mysql_tabela,
            limite=mysql_limite,
            filtro_where=mysql_filtro_where,
            verbose=verbose,
        )

        if not resultado["sucesso"]:
            raise RuntimeError(f"[agente-mysql] {resultado['erro']}")

        namespace = mysql_injetar_namespace if mysql_injetar_namespace is not None else globals()
        if namespace is not None:
            agent.injetar_no_namespace(resultado, namespace)

        df = resultado["dataframe"]
        contexto = {
            "df_variavel": resultado["variavel"],
            "df_info":     resultado["metadados"]["df_info"],
            "df_colunas":  resultado["metadados"]["colunas"],
            "df_amostra":  df.head(10).to_json(orient="records", force_ascii=False),
        }
        return resultado, contexto

    def _executar_codigo_agente(parsed: Dict) -> Dict:
        codigo = parsed.get("codigo_pandas")
        if not codigo:
            return {"sucesso": False, "resultado": ""}

        df_nome    = parsed.get("df_variavel_usada") or (df_contexto or {}).get("df_variavel")
        namespace  = mysql_injetar_namespace if mysql_injetar_namespace is not None else globals()
        exec_globals = namespace if namespace is not None else globals()
        if "__builtins__" not in exec_globals:
            exec_globals["__builtins__"] = __builtins__

        if df_nome and df_nome not in exec_globals:
            msg = f"DataFrame '{df_nome}' não encontrado no namespace."
            logger.warning(msg)
            return {"sucesso": False, "resultado": msg}

        import io, contextlib
        exec_ns = dict(exec_globals)
        exec_ns.pop("resultado", None)

        logger.debug(json.dumps({
            "location": "executar_fluxo:pre_exec",
            "agente": parsed.get("agente_id"),
            "df_variavel_usada": df_nome,
            "df_ctx_variavel": (df_contexto or {}).get("df_variavel"),
            "codigo_preview": codigo[:800],
        }, ensure_ascii=False))

        try:
            stdout = io.StringIO()
            with contextlib.redirect_stdout(stdout):
                exec(codigo, exec_ns)

            resultado_obj = exec_ns.get("resultado")
            if resultado_obj is None:
                resultado_texto = stdout.getvalue().strip()
            else:
                try:
                    resultado_texto = resultado_obj.to_string() \
                        if hasattr(resultado_obj, "to_string") else str(resultado_obj)
                except Exception:
                    resultado_texto = str(resultado_obj)

            if not resultado_texto:
                resultado_texto = stdout.getvalue().strip()
            if not resultado_texto:
                resultado_texto = "(sem saída)"

            logger.debug(json.dumps({
                "location": "executar_fluxo:post_exec",
                "agente": parsed.get("agente_id"),
                "sucesso": True,
                "resultado_preview": resultado_texto[:500],
            }, ensure_ascii=False))

            return {"sucesso": True, "resultado": resultado_texto}

        except Exception as exc:
            logger.warning(f"Erro ao executar codigo_pandas do agente {parsed.get('agente_id')}: {exc}")
            return {"sucesso": False, "resultado": f"Erro ao executar codigo_pandas: {exc}"}

    if verbose:
        print("[MAESTRO] Iniciando fluxo. Pergunta:", (pergunta[:80] + "...") if len(pergunta) > 80 else pergunta)

    # --- Passo 1+2: Análise da pergunta ---
    if verbose:
        print("[MAESTRO] Passo 1+2: Analisando pergunta (chamando LLM Maestro)...")
    skill_maestro = get_skill_by_id(skills, "maestro")
    msg_analise = (
        pergunta
        + "\n\nRetorne um JSON com uma única chave 'analise' (objeto) contendo: "
        "pergunta (string), dominios_identificados (lista de strings), tipo_resposta_esperada (uma de: factual, analítica, técnica, criativa, comparativa), "
        "complexidade (uma de: baixa, média, alta), informacao_central (string)."
    )
    resp_analise = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": skill_maestro["content"]},
            {"role": "user",   "content": msg_analise},
        ],
        response_format={"type": "json_object"},
    )
    raw_analise  = (resp_analise.choices[0].message.content or "").strip()
    analise_data = extrair_json(raw_analise)
    analise = (analise_data.get("analise") or analise_data) if isinstance(analise_data, dict) else {}
    if not analise:
        analise = {
            "pergunta": pergunta,
            "dominios_identificados": [],
            "tipo_resposta_esperada": "analítica",
            "complexidade": "média",
            "informacao_central": pergunta,
        }
    contexto_maestro = (
        f"Domínios: {analise.get('dominios_identificados', [])}; "
        f"Tipo: {analise.get('tipo_resposta_esperada', '')}; "
        f"Complexidade: {analise.get('complexidade', '')}; "
        f"Central: {analise.get('informacao_central', '')}"
    )
    if verbose:
        print("[MAESTRO] Análise obtida:", analise.get("dominios_identificados"),
              analise.get("tipo_resposta_esperada"), analise.get("complexidade"))

    if mysql_tabela:
        if verbose:
            print("[MAESTRO] Pré-carregando dados MySQL...")
        resultado_mysql, df_contexto = _preparar_contexto_mysql()
        df_variavel = df_contexto.get("df_variavel")
        if verbose:
            print(f"[MAESTRO] DataFrame disponível em '{df_variavel}'.")
        contexto_maestro = f"{contexto_maestro}; Tabela: {mysql_tabela}; DF: {df_variavel}"

    # --- Passo 3: Invocar os agentes ---
    respostas_agentes = []
    for skill_id in agentes:
        if verbose:
            print("[MAESTRO] Passo 3: Invocando agente:", skill_id)

        payload_maestro = {
            "skill_invocada": skill_id,
            "pergunta": pergunta,
            "contexto_maestro": contexto_maestro,
            "tipo_resposta_esperada": analise.get("tipo_resposta_esperada", "analítica"),
            "instrucao": "Responda estritamente dentro do seu domínio. Calcule seus scores.",
        }
        if df_contexto and skill_id in _agentes_dataframe:
            payload_maestro.update(df_contexto)
            payload_maestro["instrucao"] = "Opere em MODO DATAFRAME. Retorne codigo_pandas executável."

        raw    = invocar_agente_maestro(client, skill_id, payload_maestro, model=model, skills_list=skills)
        parsed = extrair_json(raw)

        if parsed and isinstance(parsed, dict):
            parsed.setdefault("agente_id", skill_id)

            if parsed.get("codigo_pandas"):
                exec_info        = _executar_codigo_agente(parsed)
                resultado_execucao = exec_info.get("resultado")
                if resultado_execucao:
                    prefixo       = "\n\nResultado:\n" if exec_info.get("sucesso") else "\n\nResultado (erro):\n"
                    resposta_base = parsed.get("resposta") or ""
                    parsed["resultado_execucao"] = resultado_execucao
                    parsed["resposta"]           = (resposta_base + prefixo + resultado_execucao).strip()

            respostas_agentes.append(parsed)
            if verbose:
                print("[MAESTRO]   ->", skill_id,
                      "pode_responder:", parsed.get("pode_responder"),
                      "score_final:", (parsed.get("scores") or {}).get("score_final"))
        else:
            respostas_agentes.append({
                "agente_id": skill_id, "agente_nome": skill_id,
                "pode_responder": False,
                "justificativa_viabilidade": "Resposta não veio em JSON válido.",
                "resposta": "", "scores": {},
            })
            if verbose:
                print("[MAESTRO]   ->", skill_id, "pode_responder: False (parse falhou)")

    # --- Passo 4: Montar payload do avaliador ---
    para_avaliador = [r for r in respostas_agentes if r.get("pode_responder") is True]
    if verbose:
        print("[MAESTRO] Passo 4: Respostas a enviar ao avaliador:",
              len(para_avaliador), "de", len(respostas_agentes))

    avaliacao = None
    if not para_avaliador:
        entrega_final = (
            "## Resposta do Maestro\n\n**Pergunta:** " + pergunta
            + "\n\nNenhum agente qualificado para responder. "
            "Sugestão: reformule a pergunta ou consulte um domínio mais específico."
        )
    else:
        # --- Passo 5: Invocar avaliador de coerência ---
        if verbose:
            print("[MAESTRO] Passo 5: Invocando avaliador de coerência...")
        payload_aval = {
            "pergunta_original": pergunta,
            "tipo_resposta_esperada": analise.get("tipo_resposta_esperada", "analítica"),
            "respostas_coletadas": [
                {
                    "agente_id":   r.get("agente_id"),
                    "agente_nome": r.get("agente_nome", r.get("agente_id")),
                    "resposta":    r.get("resposta", ""),
                    "scores_agente": r.get("scores") or {},
                    "limitacoes_da_resposta": r.get("limitacoes_da_resposta", ""),
                }
                for r in para_avaliador
            ],
        }
        skill_aval = get_skill_by_id(skills, "avaliador-coerencia")
        user_aval  = (
            json.dumps(payload_aval, ensure_ascii=False)
            + "\n\nRetorne APENAS um JSON com: avaliacao_completa (lista de objetos com "
            "agente_id, agente_nome, scores_avaliador, score_total, status, observacoes), "
            "ranking_final (lista de agente_id), conflitos_detectados, respostas_excluidas, threshold_utilizado."
        )
        resp_aval = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": skill_aval["content"]},
                {"role": "user",   "content": user_aval},
            ],
            response_format={"type": "json_object"},
        )
        raw_aval  = (resp_aval.choices[0].message.content or "").strip()
        avaliacao = extrair_json(raw_aval)
        if verbose:
            ranking = (avaliacao or {}).get("ranking_final", [])
            print("[MAESTRO] Avaliador retornou ranking:", ranking)
            if (avaliacao or {}).get("conflitos_detectados"):
                print("[MAESTRO]   Conflitos:", avaliacao.get("conflitos_detectados"))

        # --- Passo 6: Entrega ao usuário ---
        if verbose:
            print("[MAESTRO] Passo 6: Formatando entrega ao usuário.")
        entrega_final = _formatar_entrega(pergunta, respostas_agentes, avaliacao, para_avaliador)

    if verbose:
        print("[MAESTRO] Fluxo concluído.")
    return {
        "analise":          analise,
        "respostas_agentes": respostas_agentes,
        "avaliacao":        avaliacao,
        "entrega_final":    entrega_final,
        "resultado_mysql":  resultado_mysql,
        "df_variavel":      df_variavel,
    }


def _formatar_entrega(pergunta: str, respostas_agentes: List[Dict], avaliacao: Optional[Dict], para_avaliador: List[Dict]) -> str:
    """Gera o markdown da entrega conforme PASSO 6 do Maestro."""
    n_consultados  = len(respostas_agentes)
    aval           = avaliacao or {}
    avaliacao_completa = aval.get("avaliacao_completa") or []
    ranking        = aval.get("ranking_final") or [r.get("agente_id") for r in para_avaliador]
    id_to_resp     = {r.get("agente_id"): r for r in para_avaliador}
    ordenados      = [id_to_resp[aid] for aid in ranking if aid in id_to_resp]
    for r in para_avaliador:
        if r.get("agente_id") not in ranking:
            ordenados.append(r)
    n_qualificadas = len(ordenados)
    linhas = [
        "## Resposta do Maestro", "",
        "**Pergunta:** " + pergunta,
        f"**Agentes consultados:** {n_consultados} | **Respostas qualificadas:** {n_qualificadas}",
        "---",
    ]
    for r in ordenados:
        aid        = r.get("agente_id", "")
        nome       = r.get("agente_nome", aid)
        score_total = next(
            (item.get("score_total") for item in avaliacao_completa if item.get("agente_id") == aid),
            (r.get("scores") or {}).get("score_final", 0)
        )
        pct = round((score_total or 0) * 100)
        linhas += [f"### {nome} — {aid}", f"*Score de Confiança: {pct}%*", "", r.get("resposta", ""), "", "---"]
    linhas += ["", "### Síntese"]
    resumos = [
        (r.get("resposta", "")[:120] + "...") if len(r.get("resposta", "")) > 120 else r.get("resposta", "")
        for r in ordenados
    ]
    linhas.append("Integração das perspectivas: " + " | ".join(resumos))
    return "\n".join(linhas)


In [18]:
# PERGUNTA_TESTE = (
#     "Quais são as melhores práticas para uma startup de SaaS precificar o produto "
#     "e quais ferramentas técnicas (APIs, billing) ajudam a implementar isso?"
# )

# resultado = executar_fluxo_maestro(client=client, pergunta=PERGUNTA_TESTE, model=model, agentes=["agente-tecnico", "agente-financeiro"], verbose=True)
# print(resultado["entrega_final"])

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# CÉLULA ÚNICA — Pergunta ao Maestro e ele busca 10 linhas da tabela
# ─────────────────────────────────────────────────────────────────────────────

# ── 1. Pergunta em linguagem natural ─────────────────────────────────────────

# PERGUNTA = "Me mostre 10 linhas da tabela de servicos"

# # ── 2. Maestro identifica qual tabela buscar ──────────────────────────────────

# skill_maestro = get_skill_by_id(skills, "maestro")

# resp = client.chat.completions.create(
#     model=model,
#     messages=[
#         {"role": "system", "content": skill_maestro["content"]},
#         {"role": "user",   "content": (
#             f"{PERGUNTA}\n\n"
#             "Retorne APENAS um JSON com a chave 'tabela' contendo o nome da tabela "
#             "que o usuário quer consultar. Ex: {\"tabela\": \"vendas\"}"
#         )},
#     ],
#     response_format={"type": "json_object"},
# )

# tabela = extrair_json(resp.choices[0].message.content).get("tabela", "")
# print(f"[MAESTRO] Tabela identificada: '{tabela}'")

# # ── 3. agente-mysql busca as 10 linhas ───────────────────────────────────────

# agent = MySQLAgent(
#     host    = os.environ.get("MYSQL_HOST"),
#     banco   = os.environ.get("MYSQL_DATABASE"),
#     usuario = os.environ.get("MYSQL_USER"),
#     senha   = os.environ.get("MYSQL_PASSWORD"),
# )

# resultado = agent.carregar_tabela(tabela, limite=10, verbose=False)

# if not resultado["sucesso"]:
#     print(f"[ERRO] {resultado['erro']}")
# else:
#     df = resultado["dataframe"]
#     print(f"\n✅ Tabela '{tabela}' — 10 primeiras linhas:\n")
#     print(df.to_string(index=False))

In [20]:
# # ─────────────────────────────────────────────────────────────────────────────
# # CÉLULA: MySQL → agente-dados → executa código Pandas
# # ─────────────────────────────────────────────────────────────────────────────

# # ── Configuração ──────────────────────────────────────────────────────────────

# PERGUNTA   = "Quais os 5 serviços com maior custo_fixo e estão ativos?"
# TABELA     = "servicos"

# resultado = executar_fluxo_maestro(
#     client=client,
#     pergunta=PERGUNTA,
#     model=model,
#     agentes=["agente-dados"],
#     verbose=True,
#     mysql_host=os.environ.get("MYSQL_HOST"),
#     mysql_porta=int(os.environ.get("MYSQL_PORT", "3306")),
#     mysql_usuario=os.environ.get("MYSQL_USER"),
#     mysql_senha=os.environ.get("MYSQL_PASSWORD"),
#     mysql_banco=os.environ.get("MYSQL_DATABASE"),
#     mysql_tabela=TABELA,
#     mysql_limite=50_000,
#     mysql_filtro_where="",
#     mysql_injetar_namespace=globals(),
# )

# resp = next((r for r in resultado["respostas_agentes"] if r.get("agente_id") == "agente-dados"), {})

# print(f"\n[agente-dados] pode_responder: {resp.get('pode_responder')}")
# print(f"[agente-dados] score_final:    {resp.get('scores', {}).get('score_final')}")
# print(f"[agente-dados] {resp.get('justificativa_viabilidade')}")

# # ── Executa o código Pandas gerado ───────────────────────────────────────────

# codigo = resp.get("codigo_pandas", "")

# if not codigo:
#     print("\n[AVISO] agente-dados não retornou codigo_pandas.")
#     print("Resposta:", resp.get("resposta"))
# else:
#     print(f"\n{'─'*60}")
#     print("CÓDIGO GERADO PELO agente-dados:")
#     print('─'*60)
#     print(codigo)
#     print('─'*60)
#     print("\nRESULTADO:\n")
#     exec(codigo)   # executa no namespace atual — df_servicos já está disponível

In [21]:
PERGUNTA_TESTE = (
    "quais servicos são mais vendidos e qual o total de vendas nos ultimos 3 meses?"
)
TABELA = "os_servicos"

resultado = executar_fluxo_maestro(
    client=client,
    pergunta=PERGUNTA_TESTE,
    model=model,
    agentes=["agente-dados", "agente-negocios", "agente-financeiro"],
    verbose=True,
    mysql_host=os.environ.get("MYSQL_HOST"),
    mysql_porta=int(os.environ.get("MYSQL_PORT", "3306")),
    mysql_usuario=os.environ.get("MYSQL_USER"),
    mysql_senha=os.environ.get("MYSQL_PASSWORD"),
    mysql_banco=os.environ.get("MYSQL_DATABASE"),
    mysql_tabela=TABELA,
    mysql_limite=50_000,
    mysql_filtro_where="",
    mysql_injetar_namespace=globals(),
)
print(resultado["entrega_final"])

DEBUG:openai._base_client:Request options: {'method': 'post', 'url': '/chat/completions', 'files': None, 'idempotency_key': 'stainless-python-retry-52334e4f-4d46-4bbc-914e-1e86abf2df93', 'content': None, 'json_data': {'messages': [{'role': 'system', 'content': '---\nname: maestro\ndescription: >\n  Orquestrador central de múltiplos agentes especializados. Ative esta skill quando a pergunta do usuário\n  envolver mais de um domínio de conhecimento, ou quando precisar de respostas multidisciplinares com alta\n  precisão. O Maestro analisa a pergunta, seleciona os agentes mais adequados do registro de skills\n  disponíveis, coleta respostas com scores, envia ao Avaliador de Coerência e entrega ao usuário apenas\n  as respostas mais relevantes e consistentes. Use sempre que o usuário mencionar "agentes", "múltiplas\n  perspectivas", "orquestração", ou quando a pergunta cruzar domínios como finanças+tecnologia,\n  saúde+jurídico, dados+negócios, etc.\n---\n\n# Maestro — Orquestrador de Agen

[MAESTRO] Iniciando fluxo. Pergunta: quais servicos são mais vendidos e qual o total de vendas nos ultimos 3 meses?
[MAESTRO] Passo 1+2: Analisando pergunta (chamando LLM Maestro)...


DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7ae2b61ffda0>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x7ae2b4d29e50> server_hostname='api.openai.com' timeout=5.0
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7ae2b61567e0>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'POST']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'POST']>
DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Sat, 14 Mar 2026 22:11:42 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'CF-Ray', b'9dc69d46cd241

[MAESTRO] Análise obtida: ['Dados e BI', 'Negócios / Comercial', 'Financeiro', 'Marketing / Vendas'] analítica média
[MAESTRO] Pré-carregando dados MySQL...
[agente-mysql] Conectando ao banco 'smart'...
[agente-mysql] ✅ Tabela 'os_servicos' encontrada.
[agente-mysql] Total de linhas na tabela: 334,164
[agente-mysql] Inspecionando metadados das colunas...


DEBUG:httpcore.connection:close.started
DEBUG:httpcore.connection:close.complete


...

[MAESTRO]   -> agente-negocios pode_responder: True score_final: 0.92
[MAESTRO] Passo 3: Invocando agente: agente-financeiro


DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Sat, 14 Mar 2026 22:13:55 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'CF-Ray', b'9dc69fd7dbb870c6-GRU'), (b'CF-Cache-Status', b'DYNAMIC'), (b'Server', b'cloudflare'), (b'Strict-Transport-Security', b'max-age=31536000; includeSubDomains; preload'), (b'X-Content-Type-Options', b'nosniff'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'carsoul-pliqq2'), (b'openai-processing-ms', b'41990'), (b'openai-project', b'proj_hxilwBbr6yeydV4YtYGbjIqZ'), (b'openai-version', b'2020-10-01'), (b'x-openai-proxy-wasm', b'v0.1'), (b'x-ratelimit-limit-requests', b'5000'), (b'x-ratelimit-limit-tokens', b'4000000'), (b'x-ratelimit-remaining-requests', b'4999'), (b'x-ratelimit-remaining-tokens', b'3993764'), (b'x-ratelimit-reset-requests', b'12ms'), (b'x-ratelimit-reset-tokens', b'93ms'), (b'x-req

[MAESTRO]   -> agente-financeiro pode_responder: True score_final: 0.93
[MAESTRO] Passo 4: Respostas a enviar ao avaliador: 3 de 3
[MAESTRO] Passo 5: Invocando avaliador de coerência...


DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Sat, 14 Mar 2026 22:14:39 GMT'), (b'Content-Type', b'application/json'), (b'Transfer-Encoding', b'chunked'), (b'Connection', b'keep-alive'), (b'CF-Ray', b'9dc6a0e1eadc70c6-GRU'), (b'CF-Cache-Status', b'DYNAMIC'), (b'Server', b'cloudflare'), (b'Strict-Transport-Security', b'max-age=31536000; includeSubDomains; preload'), (b'X-Content-Type-Options', b'nosniff'), (b'access-control-expose-headers', b'X-Request-ID'), (b'openai-organization', b'carsoul-pliqq2'), (b'openai-processing-ms', b'42126'), (b'openai-project', b'proj_hxilwBbr6yeydV4YtYGbjIqZ'), (b'openai-version', b'2020-10-01'), (b'x-openai-proxy-wasm', b'v0.1'), (b'x-ratelimit-limit-requests', b'5000'), (b'x-ratelimit-limit-tokens', b'4000000'), (b'x-ratelimit-remaining-requests', b'4999'), (b'x-ratelimit-remaining-tokens', b'3994600'), (b'x-ratelimit-reset-requests', b'12ms'), (b'x-ratelimit-reset-tokens', b'81ms'), (b'x-req

[MAESTRO] Avaliador retornou ranking: ['agente-financeiro', 'agente-dados', 'agente-negocios']
[MAESTRO]   Conflitos: [{'tipo': 'Contradição direta / divergência de agregação', 'envolvidos': ['agente-dados', 'agente-negocios', 'agente-financeiro'], 'descricao': 'Existem diferenças nos totais e nas quantidades reportadas para vários servico_id (por exemplo, servico_id=2: agente-dados e agente-financeiro relatam quantidade=1486, enquanto agente-negocios relata 1362; valores monetários também diferem em diversos itens).', 'provavel_causa': 'Diferenças nos filtros e regras de agregação entre agentes — agente-negocios não menciona exclusão de registros inativos (ativo!=1) enquanto os outros dois a incluem; adicionalmente, houve diferenças de escolha de campo de valor (valor_venda_real vs fallback para valor_venda) e possivelmente pequenos ajustes de pré-processamento.', 'recomendacao': 'Reconciliar regras: decidir e padronizar (1) critério de data (data_fechamento vs created_at), (2) exclus